# 11 · Spark + RustFS via `s3a://` (Caso D)

**Teoria**: docs/08-spark-e-armazenamento-objetos.md

**Pré-requisito**: `make up-s3` (inicia o cluster Standalone do Caso B
*mais* um servidor RustFS de nó único com 4 drives / Erasure Coding RS(4,2) —
o mesmo padrão do `cdn-s3-lab`).

---

🎯 **Objetivo**: este notebook demonstra o Spark integrado a um **Object Store**
compatível com S3 — o **RustFS** — através do conector `s3a://` do Hadoop.

💡 **Mudança de paradigma**: enquanto o HDFS organiza dados em **blocos** replicados
entre DataNodes, um Object Store como o RustFS (ou Amazon S3, MinIO, etc.) armazena
dados como **objetos** em **buckets** planos, com uma API HTTP REST e sem hierarquia
de diretórios real.

📌 **Arquitetura deste caso (Caso D):**
- Cluster **Spark Standalone** (Master + Worker) — igual ao Caso B
- **RustFS** como armazenamento de objetos (compatível com S3 API)
- Conector **S3A** do Hadoop para o Spark acessar o RustFS
- **Spark Connect** (cliente leve gRPC) para submissão remota — igual ao Caso B

> 💡 O comando de inicialização do container `spark-connect` no `docker-compose.yml`
> já carrega a configuração `spark.hadoop.fs.s3a.*` apontando para
> `rustfs-server:9000` — seu cliente leve não precisa repeti-la.

In [ ]:
import sys

# Adiciona o diretório scripts/ ao path do Python para importar funções auxiliares
sys.path.insert(0, "../scripts")
from lab_utils import get_connect_session, layer_path

# Cria uma SparkSession remota via Spark Connect (gRPC)
# get_connect_session() conecta em sc://localhost:15002 — o Spark Connect Server
# O processamento real ocorre no cluster Standalone (container spark-worker)
spark = get_connect_session("11-spark-s3-rustfs")
spark

### 🧠 Como o S3A Connector funciona

O conector **S3A** (Hadoop 3.x) implementa o sistema de arquivos Hadoop (`FileSystem`
abstract interface) sobre a API S3. Quando o Spark lê `s3a://bronze/vendas`, acontece:

1. O Spark chama `FileSystem.get("s3a://bronze/vendas")`
2. O S3A Connector traduz para requisições HTTP REST:
   - `ListObjectsV2` → lista os objetos no "prefixo" bronze/vendas/
   - `GetObject` → baixa cada objeto Parquet
3. O RustFS recebe as requisições e serve os objetos
4. Os dados são distribuídos para os executores Spark

📌 **Diferenças importantes vs HDFS:**
- Sem `rename()` atômico: escrever `s3a://gold/...` requer `copy + delete`
- Sem data locality: objetos não têm "nó mais próximo"
- Sem hierarquia de diretórios real: `s3a://bronze/vendas/ano=2026/mes=07/` é
  apenas um prefixo de chave, não uma estrutura de pastas

## Inicializando a camada Bronze no S3

`make up-s3` já criou buckets vazios `bronze`/`silver`/`gold`
(`rustfs-init` no `docker-compose.yml`). Agora vamos escrever o dataset gerado
localmente em `s3a://bronze/...`.

🔁 **Fluxo de dados (diferente do HDFS):**
```
Seu host (comando via Spark Connect gRPC)
  → Spark Connect Server (container spark-connect)
    → Worker (container spark-worker)
      → lê dados do volume compartilhado /data/bronze/...
      → escreve em s3a://bronze/... via S3A Connector
        → RustFS Server (container rustfs-server)
          → 4 drives com Erasure Coding RS(4,2)
```

📌 **Note a diferença para o Caso C (HDFS)**: aqui, o Worker (dentro do container)
tem acesso direto ao volume Docker compartilhado `/data`. Nenhum upload via HttpFS
é necessário — o Worker lê os Parquet locais e escreve diretamente no RustFS.

> 💡 **O Spark Connect Server atua como um proxy**: seu notebook envia o plano
> lógico (DAG não resolvido) via gRPC, o servidor resolve e executa no cluster.

In [ ]:
# Lê os dados do volume Docker compartilhado (/data/bronze/...)
# layer_path("connect", "bronze", "vendas") resolve para /data/bronze/vendas
# Isso funciona porque spark-worker tem /data montado como bind do host
local_vendas = spark.read.parquet(layer_path("connect", "bronze", "vendas"))
local_empresas = spark.read.parquet(layer_path("connect", "bronze", "empresas"))

# Escreve os dados no RustFS via s3a://
# layer_path("s3", "bronze", "vendas") resolve para s3a://bronze/vendas
# O Spark distribui a escrita entre os Workers
local_vendas.write.mode("overwrite").parquet(layer_path("s3", "bronze", "vendas"))
local_empresas.write.mode("overwrite").parquet(layer_path("s3", "bronze", "empresas"))
print("Bronze layer written to s3a://bronze/")

### ✅ Bronze escrita no RustFS

📌 **Observação importante**: diferentemente do HDFS (onde escrevemos via HttpFS REST),
aqui a escrita acontece **dentro do container Spark Worker**, que tem acesso
direto ao RustFS via rede Docker.

Isso significa:
- ✅ Sem upload manual de arquivos
- ✅ Escrita paralelizada entre Workers
- ✅ Mais rápido que o HDFS neste setup dockerizado

> 💡 **Verificação**: abra a UI do RustFS em http://localhost:9001 para ver
> os objetos criados nos buckets `bronze`, `silver`, `gold`.

In [ ]:
# Lê de volta do RustFS para confirmar a integridade
# S3A Connector faz requisições HTTP ListObjects + GetObject para o RustFS
vendas_s3 = spark.read.parquet(layer_path("s3", "bronze", "vendas"))

# .count() força a leitura de todos os objetos Parquet do bucket
print(f"Read back from RustFS: {vendas_s3.count():,} rows")

# .show(5) exibe as primeiras linhas para verificar os dados
vendas_s3.show(5)

### 📌 Análise da leitura

✅ **Dados lidos com sucesso do RustFS!** 

Note que a leitura `s3a://...` pode ser **mais lenta** que a leitura do volume Docker
compartilhado (`/data/...`) porque:
- Cada objeto requer uma requisição HTTP REST
- O S3A Connector precisa listar objetos, depois baixar cada um
- Não há data locality (os dados podem estar em qualquer drive do RustFS)

No entanto, este custo é compensado pela **elasticidade**: um Object Store pode
armazenar **petabytes** de dados que não caberiam em volumes Docker locais.

## Partition pruning: por que é ainda mais importante no S3

Como o S3 não tem localidade de dados (docs/08), pular partições irrelevantes
antes que qualquer byte cruze a rede é mais importante aqui do que no HDFS.

🎯 **O experimento:**
1. Fazemos uma varredura **completa** em todas as partições
2. Fazemos uma leitura **filtrada** em `ano = 2026 AND mes = 7`
3. Comparamos os tempos
4. Verificamos o plano de execução com `.explain()` para confirmar o `PartitionFilters`

📌 **Por que partition pruning é mais crítico no S3?**
- No HDFS, ler dados "desnecessários" ainda é relativamente barato (data locality)
- No S3, cada byte lido cruza a rede e custa $$$ (em nuvem real, você paga por requisição)
- Pular partições = menos requisições HTTP = menos latência + menos custo

In [ ]:
import time

from pyspark.sql.functions import sum as spark_sum

# --- Varredura COMPLETA (lê TODAS as partições) ---
start = time.perf_counter()
full = vendas_s3.agg(spark_sum("valor")).collect()  # Action: dispara o Job
full_seconds = time.perf_counter() - start

# --- Varredura FILTRADA (lê APENAS as partições de julho/2026) ---
# O Spark aplica partition pruning automaticamente — ele lê apenas os
# arquivos Parquet dentro do prefixo s3a://bronze/vendas/ano=2026/mes=7/
filtered = vendas_s3.filter("ano = 2026 AND mes = 7")

# .explain() mostra o plano físico — procure por "PartitionFilters"
# Se o pruning funcionou, você verá algo como:
# PartitionFilters: [isnotnull(ano), isnotnull(mes), (ano = 2026), (mes = 7)]
filtered.explain()  # look for PartitionFilters in the plan

start = time.perf_counter()
filtered_result = filtered.agg(spark_sum("valor")).collect()  # Action
filtered_seconds = time.perf_counter() - start

# Comparação dos tempos — a diferença deve ser significativa!
print(f"Full scan:     {full_seconds:.2f}s")
print(f"Filtered scan: {filtered_seconds:.2f}s (only ano=2026/mes=7 partitions read)")

### 📊 Interpretando os resultados do partition pruning

Você deve observar que a varredura filtrada é **significativamente mais rápida**
que a completa (tipicamente 5-10× mais rápida).

📌 **Por quê?**
- Os dados estão particionados por `ano`/`mes` no S3
- Cada partição é um prefixo separado: `s3a://bronze/vendas/ano=2026/mes=7/`
- O Spark usa os metadados do Parquet para saber quais partições existem
- O filtro `ano = 2026 AND mes = 7` permite que o Spark **pule** todas as outras
  partições — sem listar objetos, sem baixar arquivos, sem ler dados

> 💡 **Dica de produção**: em ambientes de nuvem (AWS S3 real), o partition pruning
> não é apenas otimização de desempenho — é **economia de dinheiro**. Cada requisição
> S3 tem custo, e dados transferidos têm custo. Pular 90% das partições = 90% menos
> custo.
>
> 🔍 No plano `.explain()`, procure por `PartitionFilters` e `PushedFilters` —
> essas são as "dicas" que o Spark passa ao conector S3A para limitar a lista
> de objetos.

## Escrevendo as camadas Silver e Gold

Mesmo padrão medalhão (medallion) como em todos os outros casos — apenas a camada de
armazenamento mudou de `webhdfs://` para `s3a://`.

🧠 **Pipeline completo (arquitetura Medalhão):**
1. **Bronze**: dados brutos, sem transformações (já escrevemos)
2. **Silver**: dados limpos (filtramos `valor > 0`)
3. **Gold**: dados agregados e prontos para consumo (join + groupBy)

📌 **Observação**: o join com `broadcast()` envia a tabela `empresas` (pequena)
para todos os executores como uma variável broadcast — evitando shuffle.
Isso é particularmente importante no S3, onde cada shuffle adicional
significa mais leituras/escritas no object store.

In [ ]:
from pyspark.sql.functions import broadcast

# Lê empresas do RustFS (broadcast hint = envia para todos os executores via memória)
empresas_s3 = spark.read.parquet(layer_path("s3", "bronze", "empresas"))

# Camada Silver: filtra valores inválidos (valor <= 0) e persiste no S3
silver = vendas_s3.filter("valor > 0")  # Remove linhas com valor <= 0
silver.write.mode("overwrite").parquet(layer_path("s3", "silver", "vendas"))

# Camada Gold: join vendas + empresas, agrega por setor/mês e persiste
gold = (
    spark.read.parquet(layer_path("s3", "silver", "vendas"))  # Lê a Silver
    .join(broadcast(empresas_s3), "id_empresa")  # Broadcast join: envia empresas p/ todos os execs
    .groupBy("setor", "ano", "mes")  # Agrupa por setor e período
    .agg(spark_sum("valor").alias("total_vendas"))  # Soma o valor de vendas
)
gold.write.mode("overwrite").parquet(layer_path("s3", "gold", "vendas_por_setor"))

print("Gold layer written. Check the RustFS console: http://localhost:9001")
print("Pipeline Bronze → Silver → Gold concluído no S3!")

### 🏁 Pipeline completo no Caso D

✅ **Finalizamos o pipeline Medalhão inteiro usando RustFS como armazenamento!**

📌 **Resumo do que aprendemos neste notebook:**
1. Como o Spark se conecta ao **RustFS** via conector S3A (`s3a://`)
2. Como ler e escrever dados em um Object Store compatível com S3
3. A importância do **partition pruning** no S3 (vs HDFS)
4. Como o broadcast join reduz o custo de shuffle no S3
5. O pipeline Medalhão (Bronze → Silver → Gold) funciona em qualquer armazenamento

> 💡 **Próximo passo**: Vá para o Lab 12 para comparar HDFS vs S3 lado a lado
> em termos de desempenho de operações específicas (commit, scan, etc.).

In [ ]:
# Finaliza a SparkSession (desconecta do Spark Connect Server)
spark.stop()